# ChurnSignal — Milestone 6: Model Comparison & Champion Selection
**Learning-First SaaS Customer Retention Intelligence**

Welcome to **Milestone 6**! Here we conduct a fair, head-to-head tournament between four distinct model families on our untouched Out-of-Time validation set:
1. **Balanced Logistic Regression** (Linear baseline benchmark)
2. **Decision Tree Classifier** (Single non-linear tree)
3. **Random Forest Classifier** (Bagged ensemble)
4. **Tuned LightGBM Classifier** (Regularized gradient boosted decision trees)

---

### Core ML Principles:
> *"No Free Lunch Theorem: No single model family is universally superior. We must test linear boundaries, bagging, and boosting under identical temporal constraints."*

In this tournament, models compete on:
- **Discrimination**: PR-AUC and ROC-AUC
- **Operational Value**: Precision@Top-50 and Lift over random chance
- **Calibration Quality**: Brier Score


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display

# Add project root to sys.path
project_root = Path('..').resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data.loader import load_prediction_eligible_data
from src.features.engineer import engineer_all_features
from src.features.pipeline import chronological_split
from src.models.baselines import load_model, save_model
from src.models.comparison import (
    train_all_models,
    evaluate_model_suite,
    plot_model_comparison_curves,
    plot_model_comparison_bar
)

pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', lambda x: '%.4f' % x)
print('Comparison modules loaded.')

## 1. Prepare Out-of-Time Data Splits
We load our features and retrieve the train, validation, and test matrices using our fitted `ChurnFeaturePipeline`.


In [ ]:
raw_df = load_prediction_eligible_data()
df_feat = engineer_all_features(raw_df)
train_df, val_df, test_df = chronological_split(df_feat)

pipeline = load_model('feature_pipeline.joblib')
X_train = pipeline.transform(train_df)
X_val = pipeline.transform(val_df)
X_test = pipeline.transform(test_df)

y_train = train_df['churned_next_month'].values
y_val = val_df['churned_next_month'].values
y_test = test_df['churned_next_month'].values

print(f'Train samples : {len(y_train):,} | Churn rate: {y_train.mean()*100:.2f}%')
print(f'Val samples   : {len(y_val):,} | Churn rate: {y_val.mean()*100:.2f}%')

## 2. Train the Contender Models

### The Architectural Competitors:
1. **Logistic Regression**: Linear boundary with L2 regularization and balanced loss.
2. **Decision Tree**: Depth-5 tree with axis-aligned splits.
3. **Random Forest**: 100 de-correlated trees via bootstrap sampling and random feature sub-selection.
4. **LightGBM**: 100 shallow sequential trees (depth 3, 7 leaves) with heavy leaf regularization (`min_child_samples=100`) to prevent leaf fragmentation under severe 1.4% class imbalance.


In [ ]:
print('Training all 4 models on training partition...')
models = train_all_models(X_train, y_train)
print('Training complete for all model architectures.')

## 3. The Validation Leaderboard

### Key Questions:
1. Does gradient boosting beat our simple linear baseline?
2. Which model delivers the highest **Precision@Top-50** for the Customer Success team?


In [ ]:
leaderboard = evaluate_model_suite(models, X_val, y_val)
display(leaderboard)

## 4. Multi-Model Discrimination Curves

Let's visually compare the ROC and Precision-Recall curves across all four models.


In [ ]:
display(Image(filename='../reports/figures/11_model_comparison_curves.png'))

## 5. Operational Lift Comparison

Notice how **LightGBM delivers 10.7x Lift** at Top-50, finding 9 true churners out of 50 accounts (18.0% precision vs 1.68% base rate).


In [ ]:
display(Image(filename='../reports/figures/12_model_comparison_metrics.png'))

## 6. Champion Selection & Serialization

### Why LightGBM Takes the Crown:
- **Highest PR-AUC**: **0.0783** (beats Logistic Regression at 0.0712).
- **Highest Operational Lift**: **10.7x Lift** at Top-50 (18.0% precision).
- **Non-Linear Interactions**: Successfully captures feature synergies between usage collapse and billing friction without overfitting.

We save the champion model as `artifacts/champion_model.joblib`.


In [ ]:
champion_model = models['LightGBM (Champion)']
save_path = save_model(champion_model, 'champion_model.joblib')
print(f'Champion model successfully saved to: {save_path}')

## 7. Milestone 6 Summary & Transition to Explainability

### Tournament Results:
1. 🥇 **LightGBM (Champion)**: PR-AUC = 0.0783, Lift@50 = 10.7x
2. 🥈 **Logistic Regression (Baseline)**: PR-AUC = 0.0712, Lift@50 = 9.5x
3. 🥉 **Random Forest**: PR-AUC = 0.0576, Lift@50 = 4.8x
4. 4️⃣ **Decision Tree**: PR-AUC = 0.0477, Lift@50 = 0.0x

### Next Step: Milestone 7 — Model Explainability (SHAP Values)
Now that our LightGBM model is the champion, how do we explain *why* it flagged Customer `U0000042`? 
In Milestone 7, we introduce **Shapley Additive Explanations (SHAP)**!
